# Streaming Token Bucket Rate Limiting & Cascading Failure Defense

> **Lab 04 & Phase 05 Companion Exercise**  
> **Core Architecture**: Continuous Refill Token Bucket + Dual-Phase Token Reservation & Settlement + Circuit Breaker Finite State Machine + Fallback Provider Routing.

In production AI gateways, traditional HTTP rate limiting (counting requests per second) completely breaks down. A single request might consume 15 tokens or 4,000 tokens (**The Unknown Payload Problem**), and streaming responses hold connections open for tens of seconds (**The Streaming Latency Gap**). Counting tokens post-completion leads to massive burst overages and upstream provider quota bans. Furthermore, when upstream providers degrade, uncoordinated retries trigger a **Thundering Herd Collapse**. This notebook provides a complete simulation of a **Dual-Phase Streaming Token Bucket Limiter** and a production **Circuit Breaker** with Matplotlib visualizations.

In [ ]:
# %% [Setup] Colab & Environment Bootstrap
import os, sys
try:
    is_colab = 'google.colab' in str(get_ipython())
except NameError:
    is_colab = False

if is_colab:
    print('Running in Google Colab. Cloning repository...')
    !git clone --depth 1 https://github.com/dip4k/senior-engineer-to-ai-engineer-roadmap.git repo
    %cd repo
    !pip install -q -r agent-forge/requirements.txt -r notebooks/requirements-notebooks.txt
    sys.path.insert(0, os.path.abspath('agent-forge'))
else:
    # Local environment setup: locate repo root and agent-forge
    repo_root = os.path.abspath('..') if os.path.exists('../agent-forge') else os.path.abspath('.')
    af_path = os.path.join(repo_root, 'agent-forge')
    if os.path.exists(af_path) and af_path not in sys.path:
        sys.path.insert(0, af_path)
    print(f'Running locally with agent-forge in path: {af_path}')

## 1. Architectural Architecture: The Physics of LLM Rate Limiting

Why standard rate limiting fails for generative AI:

1. **The Unknown Payload Problem**: Traditional web APIs rate-limit by requests per minute (RPM). In LLMs, rate limits are enforced by tokens per minute (TPM). An RPM-only limiter allows quota exhaustion in a single burst.
2. **The Streaming Latency Gap**: Streaming responses (Server-Sent Events) take 5 to 30 seconds. If tokens are counted only after streaming ends, 50 concurrent requests can arrive, pass the gate, and trigger an upstream `HTTP 429` storm.
3. **Dual-Phase Solution**:    - **Acquire Phase**: Deduct an estimated token allocation (e.g. 2,000 tokens) upfront.
   - **Settle Phase**: When streaming ends, refund the unconsumed tokens (`estimated - actual`) back to the bucket.

```text
  [Inbound Request] ---> 1. Acquire (Reserve 2,000 Tokens)
                               |
                               v
                         2. Stream Tokens (Actual Used: 1,200)
                               |
                               v
                         3. Settle (Refund +800 Tokens to Bucket)
```

## 2. Continuous Refill Dual-Phase Token Bucket Limiter

The mathematical foundation of the continuous token bucket:
```text
Refill Rate = Capacity / 60.0  (Tokens per second)
Current Tokens(t) = min(Capacity, Tokens_prev + (t - last_time) * Refill Rate)
```

In [ ]:
import time, random
from typing import Dict, Tuple, Optional
from dataclasses import dataclass, field

@dataclass
class BucketState:
    tpm_capacity: float
    rpm_capacity: float
    current_tokens: float
    current_requests: float
    last_refill_timestamp: float = field(default_factory=time.time)

class TokenBucketLimiter:
    def __init__(self, default_rpm: int = 60, default_tpm: int = 10000):
        self.default_rpm = float(default_rpm)
        self.default_tpm = float(default_tpm)
        self.rpm_refill_rate = self.default_rpm / 60.0  # requests per sec
        self.tpm_refill_rate = self.default_tpm / 60.0  # tokens per sec
        self.buckets: Dict[str, BucketState] = {}

    def _get_or_create(self, tenant_id: str) -> BucketState:
        if tenant_id not in self.buckets:
            self.buckets[tenant_id] = BucketState(
                tpm_capacity=self.default_tpm,
                rpm_capacity=self.default_rpm,
                current_tokens=self.default_tpm,
                current_requests=self.default_rpm
            )
        return self.buckets[tenant_id]

    def _refill(self, bucket: BucketState, current_time: float):
        elapsed = current_time - bucket.last_refill_timestamp
        if elapsed <= 0:
            return
        bucket.current_requests = min(bucket.rpm_capacity, bucket.current_requests + elapsed * self.rpm_refill_rate)
        bucket.current_tokens = min(bucket.tpm_capacity, bucket.current_tokens + elapsed * self.tpm_refill_rate)
        bucket.last_refill_timestamp = current_time

    def acquire(self, tenant_id: str, estimated_tokens: int, current_time: Optional[float] = None) -> Tuple[bool, str]:
        now = current_time if current_time is not None else time.time()
        bucket = self._get_or_create(tenant_id)
        self._refill(bucket, now)

        if bucket.current_requests < 1.0:
            return False, 'RPM limit exceeded. Throttled.'
        if bucket.current_tokens < estimated_tokens:
            return False, f'TPM limit exceeded. Requested {estimated_tokens}, available {bucket.current_tokens:.0f}.'

        # Deduct upfront reservation
        bucket.current_requests -= 1.0
        bucket.current_tokens -= float(estimated_tokens)
        return True, 'OK'

    def settle(self, tenant_id: str, estimated_tokens: int, actual_tokens: int, current_time: Optional[float] = None):
        now = current_time if current_time is not None else time.time()
        bucket = self._get_or_create(tenant_id)
        self._refill(bucket, now)
        delta = float(estimated_tokens - actual_tokens)
        bucket.current_tokens = min(bucket.tpm_capacity, bucket.current_tokens + delta)

# Sanity Test
limiter = TokenBucketLimiter(default_rpm=10, default_tpm=5000)
ok, msg = limiter.acquire('tenant_1', estimated_tokens=2000)
print(f'Initial reservation (2000 tokens): Allowed={ok}, msg={msg}')
limiter.settle('tenant_1', estimated_tokens=2000, actual_tokens=1200)
print(f'Post-stream settlement (refunded 800 tokens): Remaining tokens={limiter.buckets["tenant_1"].current_tokens:.0f}')

## 3. High-Concurrency Burst Traffic Simulation

We simulate a realistic burst of 20 concurrent requests arriving over a 15-second window. We record token levels, reservation events, and settlement refunds over time.

In [ ]:
# ---------------------------------------------------------------------------
# Burst Simulation over Virtual Time (Seconds)
# ---------------------------------------------------------------------------
sim_limiter = TokenBucketLimiter(default_rpm=30, default_tpm=6000)
tenant = 'sim_tenant'

time_series = []
token_history = []
request_status = []

current_time = 0.0
requests_to_dispatch = [
    (0.5, 1500, 900),
    (0.8, 1800, 1100),
    (1.2, 2200, 1400),
    (1.5, 2000, 1300),  # Will be throttled due to TPM exhaustion
    (2.0, 1500, 950),   # Will be throttled
    (5.0, 1000, 700),   # Capacity refilled
    (8.0, 1200, 800),
    (12.0, 1500, 1000),
]

active_streams = []

for step in range(150):
    t = step * 0.1
    # Check if active streams finish and settle
    finished = [s for s in active_streams if s['finish_time'] <= t]
    for s in finished:
        sim_limiter.settle(tenant, s['est'], s['act'], current_time=t)
        active_streams.remove(s)

    # Check for new requests arriving at time t
    for req in requests_to_dispatch:
        if abs(req[0] - t) < 0.05:
            allowed, msg = sim_limiter.acquire(tenant, req[1], current_time=t)
            request_status.append({'time': t, 'allowed': allowed, 'tokens': req[1]})
            if allowed:
                active_streams.append({'finish_time': t + 2.0, 'est': req[1], 'act': req[2]})

    b = sim_limiter._get_or_create(tenant)
    sim_limiter._refill(b, t)
    time_series.append(t)
    token_history.append(b.current_tokens)

print(f'Burst simulation complete. Tracked {len(time_series)} time steps.')
print(f'Accepted requests: {sum(1 for r in request_status if r["allowed"])} / {len(requests_to_dispatch)}')

## 4. Cascading Failure Defense & Circuit Breakers

When an upstream foundation model provider degrades (returning 503 Overloaded or 429 RateLimitError), callers without a circuit breaker retry endlessly, creating a **Thundering Herd Collapse**.

The **Circuit Breaker Finite State Machine (FSM)**:
- `CLOSED`: Healthy. Requests flow through normally. Consecutive errors are counted.
- `OPEN`: Failure threshold exceeded (e.g. 3 consecutive errors). Fails fast or routes to fallback model.
- `HALF-OPEN`: Cooldown period elapsed (e.g. 5 seconds). Canary probe request allowed. If successful, resets to `CLOSED`; if failed, re-opens.

In [ ]:
# ---------------------------------------------------------------------------
# Circuit Breaker Finite State Machine (FSM)
# ---------------------------------------------------------------------------
from enum import Enum

class BreakerState(str, Enum):
    CLOSED = 'CLOSED'
    OPEN = 'OPEN'
    HALF_OPEN = 'HALF_OPEN'

class CircuitBreaker:
    def __init__(self, failure_threshold: int = 3, cooldown_seconds: float = 4.0):
        self.failure_threshold = failure_threshold
        self.cooldown_seconds = cooldown_seconds
        self.state = BreakerState.CLOSED
        self.failure_count = 0
        self.last_state_change = time.time()
        self.state_history = []

    def record_success(self, current_time: float):
        if self.state == BreakerState.HALF_OPEN:
            self.state = BreakerState.CLOSED
            self.failure_count = 0
            self.last_state_change = current_time
            self.state_history.append((current_time, BreakerState.CLOSED, 'Canary Probe Succeeded'))
        elif self.state == BreakerState.CLOSED:
            self.failure_count = 0

    def record_failure(self, current_time: float):
        self.failure_count += 1
        if self.state in [BreakerState.CLOSED, BreakerState.HALF_OPEN] and self.failure_count >= self.failure_threshold:
            self.state = BreakerState.OPEN
            self.last_state_change = current_time
            self.state_history.append((current_time, BreakerState.OPEN, f'Failure threshold {self.failure_threshold} reached'))

    def can_attempt(self, current_time: float) -> Tuple[bool, str]:
        if self.state == BreakerState.CLOSED:
            return True, 'Normal Execution'
        if self.state == BreakerState.OPEN:
            if current_time - self.last_state_change >= self.cooldown_seconds:
                self.state = BreakerState.HALF_OPEN
                self.last_state_change = current_time
                self.state_history.append((current_time, BreakerState.HALF_OPEN, 'Cooldown elapsed; testing canary probe'))
                return True, 'Canary Probe Allowed'
            return False, 'Circuit OPEN: Fail fast or route to fallback model'
        return True, 'Canary Probe Active'

# Test Circuit Breaker transitions
cb = CircuitBreaker(failure_threshold=3, cooldown_seconds=4.0)
t = 0.0
cb.record_failure(t)
cb.record_failure(t + 0.5)
cb.record_failure(t + 1.0)  # Trips breaker to OPEN
print(f'State after 3 failures: {cb.state.value}')

# Test fail fast
can_call, msg = cb.can_attempt(t + 2.0)
print(f'Attempt during cooldown (t=2.0): Allowed={can_call}, reason={msg}')

# Test transition to HALF_OPEN after cooldown
can_call, msg = cb.can_attempt(t + 5.0)
print(f'Attempt after cooldown (t=5.0): Allowed={can_call}, State={cb.state.value}')

# Canary succeeds -> reset to CLOSED
cb.record_success(t + 5.2)
print(f'State after canary success: {cb.state.value}')

## 5. Visualizing Rate Limiting Dynamics & Circuit Breaker Transitions

We plot the dual dynamics:
1. **Token Capacity Curve**: Showing reservation depletion, throttling barrier, and settlement replenishment.
2. **Circuit Breaker State Machine**: Visualizing Closed -> Open -> Half-Open -> Closed transitions.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

# Panel 1: Token Bucket Capacity
ax1.plot(time_series, token_history, label='Available Token Capacity', color='#1f77b4', linewidth=2)
ax1.axhline(6000, color='gray', linestyle='--', alpha=0.6, label='Max TPM Capacity (6,000)')
ax1.axhline(0, color='red', linestyle=':', alpha=0.8, label='Exhaustion Ceiling (0)')

# Mark accepted vs throttled requests
for req in request_status:
    if req['allowed']:
        ax1.scatter(req['time'], 5800, color='green', marker='^', s=80, label='Accepted' if req['time'] == 0.5 else '')
    else:
        ax1.scatter(req['time'], 5800, color='red', marker='x', s=100, linewidth=2, label='Throttled 429' if req['time'] == 1.5 else '')

ax1.set_ylabel('Tokens in Bucket', fontsize=10)
ax1.set_title('Dual-Phase Streaming Token Bucket Rate Limiting (Reservation & Settlement)', fontsize=11, fontweight='bold')
ax1.legend(loc='lower left', fontsize=9)
ax1.grid(True, linestyle=':', alpha=0.6)

# Panel 2: Circuit Breaker Simulation Trace
cb_sim_times = [0, 1.0, 1.0, 5.0, 5.0, 8.0, 8.0, 14.0]
cb_states = [1, 1, 0, 0, 0.5, 0.5, 1, 1]  # 1=Closed, 0=Open, 0.5=Half-Open

ax2.step(cb_sim_times, cb_states, where='post', color='#ff7f0e', linewidth=2.5, label='Circuit State')
ax2.fill_between(cb_sim_times, cb_states, step='post', alpha=0.15, color='#ff7f0e')
ax2.set_yticks([0, 0.5, 1])
ax2.set_yticklabels(['OPEN\n(Fallback)', 'HALF-OPEN\n(Canary)', 'CLOSED\n(Healthy)'], fontsize=9)
ax2.set_xlabel('Time (Seconds)', fontsize=10)
ax2.set_ylabel('Circuit State', fontsize=10)
ax2.set_title('Circuit Breaker FSM: Provider Outage -> Open -> Half-Open -> Closed', fontsize=11, fontweight='bold')
ax2.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

## 6. Interactive Katas: Exponential Backoff with Full Jitter

When an agent retries after an upstream 429 error, fixed retry intervals cause synchronized request waves. **Exponential Backoff with Full Jitter** spreads requests evenly across the window:
```text
Sleep = Uniform(0, min(max_backoff, base_backoff * (2 ** attempt)))
```

In [ ]:
# ---------------------------------------------------------------------------
# Kata: Exponential Backoff with Full Jitter
# ---------------------------------------------------------------------------
def compute_jitter_backoff(attempt: int, base: float = 0.5, max_b: float = 8.0) -> float:
    temp = min(max_b, base * (2 ** attempt))
    sleep_time = random.uniform(0, temp)
    return sleep_time

print('Simulated retry sleep times for attempts 0 to 5:')
for attempt in range(6):
    samples = [f'{compute_jitter_backoff(attempt):.2f}s' for _ in range(4)]
    print(f'Attempt {attempt}: {samples}')

## 7. Summary & Production LLMOps Checklist

- **Dual-Phase Token Reservation**: Always deduct tokens upfront before streaming starts.
- **Post-Stream Reconciliation**: Settle actual token counts against reservations to maintain precise tenant balances.
- **Circuit Breakers for Upstream Outages**: Fail fast or divert to fallback models when error thresholds are exceeded.
- **Full Jitter Retries**: Avoid fixed sleep intervals to eliminate the thundering herd.